# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 1.0 ,2.0 ,3.0 и 0, 1, 1 потому что деление / возвращает float тип данных а деление без остатка // возвращает int тип данных в меньшую сторону
2. 4 4 4 m будет матрицей едениц 3 на 4 размера axis=1 суммирует по строкам и так как каждой строке 4 единиц будет три четвертки
3. 0 2 4 первый массив будет [0 1 2] второй [0 1 2] и после reshape [0 1 2] и если суммировать каждый элемент каждого массива суммируется с тем же элементом у каторого тот же индекс
4. 6 sqrt() возвращает корни элементов а значит вернет [1 2 3] массив и их сумма равна 6
5. 7,5 создасться массив от 1 до 4 включтельно [1 2 3 4] и потом создасться их квараты [1 4 9 16] и их средняя значение равно 7,5

In [2]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [4]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [5]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


In [6]:
print(days)

[ 1.  2.  3. ... nan  2.  2.]


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [7]:
# 1
row=clean.sum()
print(f"Количество строк: {row}")

sep=amount[(clean) & (month=="2026-09")].sum()
oct=amount[(clean) & (month=="2026-10")].sum()

print(f"Выручка за сентябрь: {sep}")
print(f"Выручка за октабрь: {oct}")
print(f"Изменение: {(oct-sep)/sep:+.2%}")

Количество строк: 4739
Выручка за сентябрь: 35243410.0
Выручка за октабрь: 34419520.0
Изменение: -2.34%


Да совпадает с ДЗ 2

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [8]:
# 2
unique_cat=np.unique(cat)
unique_month=np.unique(month)

table_1=np.array([[amount[clean & (cat==c) & (month==m)].sum() for m in unique_month] for c in unique_cat])

for c, r in zip(unique_cat, table_1):
    print(f"{c:<12} {r[0]:<14} {r[1]:<14}")

print(f"Лучшая категория: {unique_cat[table_1.sum(axis=1).argmax()]}")

g = table_1[:, 1] / table_1[:, 0] - 1

print(f"Сильнее всего упавшая: {unique_cat[g.argmin()]}")

Дом          4997730.0      5125790.0     
Книги        1065300.0      1140910.0     
Красота      2610950.0      2678990.0     
Одежда       9167100.0      8505730.0     
Электроника  17402330.0     16968100.0    
Лучшая категория: Электроника
Сильнее всего упавшая: Одежда


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [9]:
# 3
table_2=np.array([[(clean & (cat==c) & (month==m)).sum() for m in unique_month] for c in unique_cat])

for c, r in zip(unique_cat, table_2):
    print(f"{c:<12} {r[0]:<14} {r[1]:<14}")

print(f"Самая высокий средний чек за октябрь: {unique_cat[(table_1[:,1]/table_2[:,1]).argmax()]}")

Дом          476            476           
Книги        265            299           
Красота      422            435           
Одежда       678            652           
Электроника  531            505           
Самая высокий средний чек за октябрь: Электроника


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [10]:
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)

table_3= table / table.sum(axis=0)
print(table_3.sum(axis=0))
print(f"Измение доли выручки Шымкента: {(table_3[5,1]-table_3[5,0])/table_3[5,0]:.2%}")

Актобе        2,576,960    2,540,420
Алматы       14,501,070   14,643,020
Астана        9,034,220    8,850,530
Атырау          923,990      969,860
Караганда     3,648,520    3,371,680
Шымкент       4,558,650    4,044,010
(6, 2)
[1. 1.]
Измение доли выручки Шымкента: -9.17%


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [11]:
# 5
change=table[:,1]-table[:,0]
table_5=change/change.sum()
for c, d, s in zip(cities, change, table_5):
    print(f"{c:<10} {d:>+12,.0f} {s:>8.1%}")

Актобе          -36,540     4.4%
Алматы         +141,950   -17.2%
Астана         -183,690    22.3%
Атырау          +45,870    -5.6%
Караганда      -276,840    33.6%
Шымкент        -514,640    62.5%


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [12]:
# 6
share = table / table.sum(axis=1, keepdims=True)
print(share.sum(axis=1))
share

[1. 1. 1. 1. 1. 1.]


array([[0.50357019, 0.49642981],
       [0.49756469, 0.50243531],
       [0.50513538, 0.49486462],
       [0.48788975, 0.51211025],
       [0.51971739, 0.48028261],
       [0.52991168, 0.47008832]])

table имеет форму (6, 2), а table.sum(axis=1) имеет форму (6,). При broadcasting numpy выравнивает формы с конца и сравнивает 2 с 6. Числа не равны, и ни одно из них не 1, отсюда ошибка

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [15]:
# 7
z=(amount[clean]-amount[clean].mean())/amount[clean].std()
print(f"Количество:{(abs(z)>3).sum()}")
print(f"Доля |z|>3:{(abs(z)>3).sum()/len(z)}")
print(f"Проверька z=0: {z.mean()}, Проверька z=1 {z.std()}")

Количество:87
Доля |z|>3:0.018358303439544206
Проверька z=0: 7.04695264417065e-17, Проверька z=1 1.0


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [14]:
# 8
clean_days= clean & ~np.isnan(days)
X=np.column_stack([amount[clean_days],days[clean_days]])
x_scaled=(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print(x_scaled.shape)

print(f"Minimum: {x_scaled.min(axis=0)}")
print(f"Maximum: {x_scaled.max(axis=0)}")

(4595, 2)
Minimum: [0. 0.]
Maximum: [1. 1.]


Потому что на диапазоны amount и days разные и для модели без масштабирование будет важнее amount так как она больше чем days а после масштабирование оба массива честно звечаны